# **Maestría en Inteligencia Artificial Aplicada**

## Curso: **Procesamiento de Lenguaje Natural**

### Tecnológico de Monterrey

### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipo - Semanas 4 y 5**

### **Vectores Embebidos de HuggingFace**

#### **Nombres y matrículas de los integrantes del equipo:**

**GRUPO 61**

*   Jhonatan Marin Salazar - A01797351
*   Julián Malo Romero - A01797459
*   Gerardo Limón Cabrera - A01283060

In [ ]:
# 1. Instalación de la librería de HuggingFace (Ejecutar al menos una vez en tu entorno)
!pip install sentence-transformers

# 2. Importación de librerías requeridas
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer

# 3. Descarga de recursos de NLTK
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')



En esta actividad deberás utilizar los datos de tres archivos que se encuentran en el repositorio de la UCI llamados **amazon_cells_labelled.txt**, **imdb_labelled.txt** y   **yelp_labelled.txt**. Cada uno de estos archivos corresponden a comentarios de usuarios que adquirieron un celular a través de la plataforma de Amazon, de comentarios que dejaron usuarios sobre palículas y series en la plataforma de IMDb y sobre servicios de comida dejados en la plataforma de Yelp.

La información del problema y de los archivos están basados en el repositorio de la UCI cuya liga es la siguiente:

https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences



# **Pregunta - 1:**



Descarga los 3 archivos de la plataforma de la UCI indicado previamente y genera un nuevo DataFrame de Pandas con ellos.

**Llama simplemente "df" a dicho DataFrame.**




In [ ]:
# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

from google.colab import drive
import os
import glob
import zipfile
import csv

drive.mount('/content/drive')

# Definición de la ruta de los archivos
ruta_drive = '/content/drive/MyDrive/MNA/NLP/sentiment labelled sentences'
extract_path = '/content/sentiment_labelled_sentences'
os.makedirs(extract_path, exist_ok=True)

txt_en_drive = glob.glob(os.path.join(ruta_drive, '*.txt'))

if len(txt_en_drive) >= 3:
    ruta = ruta_drive
else:
    zip_files = glob.glob(os.path.join(ruta_drive, '*.zip'))
    if len(zip_files) == 0:
        raise FileNotFoundError('No se encontraron archivos .txt ni .zip en la ruta: ' + ruta_drive)
    with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    matches = glob.glob(os.path.join(extract_path, '**', 'amazon_cells_labelled.txt'), recursive=True)
    if len(matches) == 0:
        raise FileNotFoundError('No se encontró amazon_cells_labelled.txt después de extraer el .zip')
    ruta = os.path.dirname(matches[0])

# Carga de los archivos
dfa = pd.read_csv(os.path.join(ruta, 'amazon_cells_labelled.txt'), sep='\t', names=['review','label'], header=None, encoding='utf-8')
dfy = pd.read_csv(os.path.join(ruta, 'yelp_labelled.txt'), sep='\t', names=['review','label'], header=None, encoding='utf-8')
dfi = pd.read_csv(os.path.join(ruta, 'imdb_labelled.txt'), sep='\t', names=['review','label'], header=None, encoding='utf-8', quoting=csv.QUOTE_NONE)

# Generación de un solo DataFrame
df = pd.concat([dfa, dfy, dfi], ignore_index=True)

# Despliegue del tamaño del DataFrame df
print("Tamaño del DataFrame 'df':", df.shape)
print("Verificación de registros por fuente:")
print("- Amazon:", dfa.shape[0])
print("- Yelp:", dfy.shape[0])
print("- IMDb:", dfi.shape[0])

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# Verifiquemos la información del DataFrame:

df.info()

In [ ]:
# Y mostremos sus primeros registros:

df.head()

# **Pregunta - 2:**

Proceso de limpieza. Aplica el proceso de limpieza que consideres adecuado.











In [ ]:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********
X = df['review'].values
y = df['label'].values

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
# Es crucial conservar palabras de negación en análisis de sentimiento
stop_words.difference_update({'no', 'not', 'nor', "didn't", "doesn't", "isn't", "aren't", "wasn't", "weren't", "haven't", "hasn't", "hadn't", "won't", "wouldn't", "don't", "doesn't", "didn't", "can't", "couldn't", "shouldn't", "mightn't", "mustn't"})

Xclean = []

for review in X:
    # Convertir a minúsculas y eliminar caracteres no alfabéticos
    review_clean = re.sub(r'[^a-zA-Z\s]', '', review.lower())
    # Tokenización simple y limpieza
    tokens = review_clean.split()
    # Lematización y eliminación de stopwords
    tokens_clean = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    # Reconstrucción del comentario
    Xclean.append(" ".join(tokens_clean))


# *********** Aquí termina la sección de agregar código *************

In [ ]:
# Despleguemos los primeros comentarios después de tu proceso de limpieza:

for x in Xclean[0:5]:
  print(x)


# **Pregunta - 3:**



Realicemos una partición aleatoria con los porcentajes que consideres más adecuados. Utiliza una semilla para su reproducibilidad.

In [ ]:
# ************* Inicia la sección de agregar código:*****************************

# Partición Train (70%) y Temp (30%)
X_train_raw, X_temp, ytrain, y_temp = train_test_split(Xclean, y, train_size=0.70, random_state=42, shuffle=True, stratify=y)

# Partición Val (15%) y Test (15%) a partir de Temp (50% de 30%)
X_val_raw, X_test_raw, yval, ytest = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, shuffle=True, stratify=y_temp)

# *********** Termina la sección de agregar código *************


# verificemos las dimensiones obtenidas:
print('Dimensiones de los conjuntos:')
print('X_train, ytrain:', len(X_train_raw), len(ytrain))
print('X_val, yval:', len(X_val_raw), len(yval))
print('X_test, ytest:', len(X_test_raw), len(ytest))

# **Pregunta - 4:**




### **Construye tu vocabulario a continuación utilizando solamente el conjunto de Train:**


In [ ]:
# a. Usa el conjunto de entrenamiento para generar tu vocabulario
#    con un tamaño que consideres adecuado: una longitud mínima de 2 caracteres y una frecuencia mínima de aparición de 2 veces en el corpus

# 1. Importación de la librería necesaria para el conteo
from collections import Counter

# a. Generación del vocabulario con conjunto de entrenamiento
todas_las_palabras = " ".join(X_train_raw).split()
frecuencia_palabras = Counter(todas_las_palabras)

# Filtrado por longitud mínima (2) y frecuencia mínima (2)
vocabulario_filtrado = {palabra for palabra, freq in frecuencia_palabras.items() if len(palabra) >= 2 and freq >= 2}

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# b. Indica el tamaño del vocabulario generado.

print('Longitud del vocabulario generado:')

# ******* Inicia la sección de agregar código: ***********

# Imprimo el tamaño de la estructura de datos generada tras aplicar los filtros
print('Tamaño del vocabulario generado:', len(vocabulario_filtrado))

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# c. Con el vocabulario generado, filtra los conjuntos de entrenamiento,
#    validación y prueba para que todos los comentarios usen solamente las
#    palabras de este vocabulario.
#    Llamar train_X, val_X y test_X a estos tres conjuntos.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Defino una función auxiliar para filtrar los tokens de cada comentario
def filtrar_comentario(comentario, vocabulario):
    tokens = comentario.split()
    return [word for word in tokens if word in vocabulario]

# Aplico el filtrado iterando sobre los conjuntos crudos que creé en la pregunta 3
train_X = [filtrar_comentario(x, vocabulario_filtrado) for x in X_train_raw]
val_X = [filtrar_comentario(x, vocabulario_filtrado) for x in X_val_raw]
test_X = [filtrar_comentario(x, vocabulario_filtrado) for x in X_test_raw]

# *********** Aquí termina la sección de agregar código *************


In [ ]:
# Vemos el resultado de los primeros comentarios del conjunto de validación:

for ss in val_X[0:5]:
  print(ss)

# **Pregunta - 5:**

Incluye tus comentarios sobre cada modelo de HuggingFace indicado.

### ++++++++ Inicia la sección de agregar texto: +++++++++++

* **a) bge-base-en-v1.5**
Investigamos las especificaciones de este modelo y noto que pertenece a la familia de BAAI. Es un modelo base que genera vectores de 768 dimensiones. Selecciono mentalmente este modelo por su excelente equilibrio; ofrece un balance ideal entre velocidad computacional y precisión semántica para tareas de recuperación de texto.

* **b) bge-large-en-v1.5**
Revisamos esta versión ampliada del modelo anterior. Identifico que produce embeddings con 1024 dimensiones, lo que significa que requiere mayor capacidad de cómputo y memoria. Sin embargo, compensa este coste generando representaciones mucho más ricas y detalladas, útiles si el problema de clasificación fuera extremadamente complejo.

* **c) e5-base-v2**
Analizamos este modelo desarrollado por Microsoft. Observo que se entrena con un enfoque contrastivo profundo y tiene la particularidad de requerir "pre-textos" (como los prefijos "query:" o "passage:") para entender cómo mapear la sentencia. Demuestra resultados muy robustos al clasificar documentos bien estructurados en inglés.

### ++++++++ Termina la sección de agregar texto: +++++++++++

# **Pregunta - 6:**

In [ ]:
# a) Cargar el modelo de embeddings de HuggingFace seleccionado:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Importo la librería si no se hizo en la celda inicial
from sentence_transformers import SentenceTransformer

# Instancio y descargo el modelo 'bge-base-en-v1.5' por su buen equilibrio entre rendimiento y coste
modelo_hf = SentenceTransformer('BAAI/bge-base-en-v1.5')

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# b) Primeros 3 elementos clave:valor del diccionario generado.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Convierto mi vocabulario filtrado a una lista iterable
lista_vocabulario = list(vocabulario_filtrado)

# Obtengo los vectores continuos para todas las palabras en un solo paso
vectores = modelo_hf.encode(lista_vocabulario, show_progress_bar=False)

# Construyo el diccionario clave-valor mapeando cada palabra con su vector correspondiente
diccionario_embebidos = {palabra: vectores[i] for i, palabra in enumerate(lista_vocabulario)}

# Imprimo los primeros 3 elementos (recorto los vectores a 5 dimensiones por legibilidad visual)
print("Muestro los primeros 3 elementos clave:valor generados:")
for i, (k, v) in enumerate(diccionario_embebidos.items()):
    if i < 3:
        print(f"Palabra: '{k}' -> Vector (dim: {v.shape[0]}): {v[:5]}...")

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# c) Tamaño del diccionario generado:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Imprimo la longitud final del diccionario para confirmar que coincide con el vocabulario
print(f"Tamaño total del diccionario generado: {len(diccionario_embebidos)} elementos")

# *********** Aquí termina la sección de agregar código *************


# **Pregunta - 7:**




Generamos los vectores embebidos a partir de los conjuntos de entrenamiento, validación y prueba y con las características indicadas en el archivo PDF.

Los llamaremos trainEmb, valEmb y testEmb, respectivamente.


In [ ]:
# a) Comentarios con vectores embebidos.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Defino una función que promedia matemáticamente los vectores de los tokens presentes en el comentario
def promediar_vectores(lista_tokens, diccionario, dim=768):
    vectores_validos = [diccionario[token] for token in lista_tokens if token in diccionario]
    if not vectores_validos:
        # Retorno un vector de ceros si el comentario se quedó sin palabras válidas
        return np.zeros(dim)
    return np.mean(vectores_validos, axis=0)

# Genero los nuevos conjuntos promediando los vectores de cada comentario
trainEmb = np.array([promediar_vectores(tokens, diccionario_embebidos) for tokens in train_X])
valEmb = np.array([promediar_vectores(tokens, diccionario_embebidos) for tokens in val_X])
testEmb = np.array([promediar_vectores(tokens, diccionario_embebidos) for tokens in test_X])

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# b) Dimensiones de los conjuntos trainEmb, valEmb y testEmb.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Verifico las matrices generadas confirmando que tengan 768 columnas (dimensiones)
print('Dimensiones trainEmb:', trainEmb.shape)
print('Dimensiones valEmb:', valEmb.shape)
print('Dimensiones testEmb:', testEmb.shape)

# *********** Aquí termina la sección de agregar código *************

# **Pregunta - 8:**

In [ ]:
# Número de tokens generados al obtener cada uno de los conjuntos trainEmb, valEmb y testEmb.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Calculo la sumatoria de las longitudes de las listas de tokens utilizadas
tokens_train = sum(len(tokens) for tokens in train_X)
tokens_val = sum(len(tokens) for tokens in val_X)
tokens_test = sum(len(tokens) for tokens in test_X)

# Reporto las cantidades obtenidas
print(f'Total de tokens utilizados en Train: {tokens_train}')
print(f'Total de tokens utilizados en Validation: {tokens_val}')
print(f'Total de tokens utilizados en Test: {tokens_test}')

# *********** Aquí termina la sección de agregar código *************

# **Pregunta - 9:**



Entrenamiento y reporte de los modelos de Regresión Logística y Bosque Aleatorio (Random Forest).


In [ ]:
# 9a) REGRESIÓN LOGÍSTICA:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Instancio y entreno el modelo de Regresión Logística
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_model.fit(trainEmb, ytrain)

# Calculo la exactitud para entrenamiento y validación
acc_t_lr = lr_model.score(trainEmb, ytrain)
acc_v_lr = lr_model.score(valEmb, yval)

# Imprimo resultados y el reporte de clasificación
print("--- REGRESIÓN LOGÍSTICA (Vectores Promediados) ---")
print(f"Exactitud en Train: {acc_t_lr:.4f}")
print(f"Exactitud en Val:   {acc_v_lr:.4f}")
print(f"Diferencia: {abs(acc_t_lr - acc_v_lr)*100:.2f}%\n")
print(classification_report(yval, lr_model.predict(valEmb)))

# *********** Aquí termina la sección de agregar código *************
# *********** Aquí termina la sección de agregar código *************


In [ ]:
# 9b) BOSQUE ALEATORIO (Random Forest):

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Instancio y entreno el modelo limitando la profundidad para evitar sobreajuste severo
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(trainEmb, ytrain)

# Calculo la exactitud para entrenamiento y validación
acc_t_rf = rf_model.score(trainEmb, ytrain)
acc_v_rf = rf_model.score(valEmb, yval)

# Imprimo resultados y el reporte de clasificación
print("--- BOSQUE ALEATORIO (Vectores Promediados) ---")
print(f"Exactitud en Train: {acc_t_rf:.4f}")
print(f"Exactitud en Val:   {acc_v_rf:.4f}")
print(f"Diferencia: {abs(acc_t_rf - acc_v_rf)*100:.2f}%\n")
print(classification_report(yval, rf_model.predict(valEmb)))

# *********** Aquí termina la sección de agregar código *************

Al analizar los resultados de la primera metodología, basada en promediar los vectores embebidos de cada palabra individual, se puede comparar el comportamiento de Regresión Logística y Bosque Aleatorio a partir de las métricas impresas en las celdas anteriores.

- Regresión Logística: suele mostrar un comportamiento estable cuando los vectores promedio ya separan parcialmente las clases positiva y negativa. Para validar que no exista sobreentrenamiento, se debe revisar que la diferencia entre entrenamiento y validación no sea demasiado alta.

- Bosque Aleatorio: puede alcanzar buen desempeño en entrenamiento, pero también puede presentar mayor riesgo de sobreajuste, especialmente cuando trabaja con representaciones densas y pocas observaciones en comparación con la dimensionalidad de los vectores.

En general, esta primera metodología permite comprobar que los embeddings de palabras pueden utilizarse para representar comentarios completos. Sin embargo, al promediar los vectores se pierde parte del orden, la estructura y el contexto original del enunciado.

# **Pregunta - 10**

**Proceso basado en modelos Preentrenados**

In [ ]:
# 10a) Partición.:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Reasigno los conjuntos de texto crudo obtenidos en el Ejercicio 3
X_original = df['review'].astype(str).values
y_original = df['label'].values

X_train, X_temp_full, y_train, y_temp_full = train_test_split(X_original, y_original, train_size=0.70, random_state=42, shuffle=True, stratify=y_original)
X_val, X_test, y_val, y_test = train_test_split(X_temp_full, y_temp_full, test_size=0.50, random_state=42, shuffle=True, stratify=y_temp_full)

# Despliego las dimensiones confirmando los 3000 registros
print('Dimensiones (comentarios en crudo):')
print('X_train:', len(X_train))
print('X_val:', len(X_val))
print('X_test:', len(X_test))

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# 10b) Vectores embebidos:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

try:
    tokenizer = modelo_hf.tokenizer
except AttributeError:
    tokenizer = modelo_hf._first_module().tokenizer

tokens_train_hf = tokenizer(list(X_train), padding=False, truncation=True, return_attention_mask=False)
tokens_val_hf = tokenizer(list(X_val), padding=False, truncation=True, return_attention_mask=False)
tokens_test_hf = tokenizer(list(X_test), padding=False, truncation=True, return_attention_mask=False)

num_tokens_train = sum(len(ids) for ids in tokens_train_hf['input_ids'])
num_tokens_val = sum(len(ids) for ids in tokens_val_hf['input_ids'])
num_tokens_test = sum(len(ids) for ids in tokens_test_hf['input_ids'])

print('Tokens generados por el tokenizer de HuggingFace:')
print('Train:', num_tokens_train)
print('Validation:', num_tokens_val)
print('Test:', num_tokens_test)

# Codifico directamente los enunciados completos pasando el texto al modelo de HuggingFace
trainEmb_full = modelo_hf.encode(list(X_train), show_progress_bar=False)
valEmb_full = modelo_hf.encode(list(X_val), show_progress_bar=False)
testEmb_full = modelo_hf.encode(list(X_test), show_progress_bar=False)

# Indico los "tokens" o más bien, los comentarios procesados directamente
print('Cantidad de vectores de sentencias generados:')
print(f"Train: {len(trainEmb_full)} arreglos de {trainEmb_full.shape[1]} dimensiones")
print(f"Val:   {len(valEmb_full)} arreglos de {valEmb_full.shape[1]} dimensiones")
print(f"Test:  {len(testEmb_full)} arreglos de {testEmb_full.shape[1]} dimensiones")

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# 10c) REGRESIÓN LOGÍSTICA.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Aplico Regresión Logística sobre los vectores del enunciado completo
lr_full = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_full.fit(trainEmb_full, y_train)

acc_t_lr2 = lr_full.score(trainEmb_full, y_train)
acc_v_lr2 = lr_full.score(valEmb_full, y_val)

print("--- LR (Enunciado Completo) ---")
print(f"Train Acc: {acc_t_lr2:.4f}")
print(f"Val Acc:   {acc_v_lr2:.4f}")
print(f"Diferencia: {abs(acc_t_lr2 - acc_v_lr2)*100:.2f}%\n")
print(classification_report(y_val, lr_full.predict(valEmb_full)))

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# 10d) BOSQUE ALEATORIO.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Aplico Random Forest sobre los vectores del enunciado completo
rf_full = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_full.fit(trainEmb_full, y_train)

acc_t_rf2 = rf_full.score(trainEmb_full, y_train)
acc_v_rf2 = rf_full.score(valEmb_full, y_val)

print("--- RF (Enunciado Completo) ---")
print(f"Train Acc: {acc_t_rf2:.4f}")
print(f"Val Acc:   {acc_v_rf2:.4f}")
print(f"Diferencia: {abs(acc_t_rf2 - acc_v_rf2)*100:.2f}%\n")
print(classification_report(y_val, rf_full.predict(valEmb_full)))

# *********** Aquí termina la sección de agregar código *************

Verificación y Comparación de Resultados:

Al extraer los vectores embebidos proporcionándole al modelo preentrenado de HuggingFace el comentario en su totalidad, se aprovecha mejor la información contextual del enunciado completo.

- Regresión Logística: permite evaluar si los embeddings del enunciado completo separan adecuadamente los comentarios positivos y negativos. Se debe revisar la diferencia entre entrenamiento y validación para confirmar si el modelo generaliza correctamente.

- Bosque Aleatorio: puede mejorar respecto al enfoque de promedio de palabras, aunque también puede presentar señales de sobreentrenamiento si la diferencia entre train y validation es mayor que la tolerancia definida.

Comparación Parte I vs Parte II: El método basado en vectores de enunciados completos suele ser más sólido porque no depende únicamente del promedio de palabras. En este segundo enfoque, el modelo preentrenado conserva mejor relaciones semánticas y parte del contexto de la oración, mientras que el promedio de embeddings pierde información del orden y de la estructura del texto.

# **Pregunta - 11:**

In [ ]:
# Reporte del mejor modelo y partición con el conjunto de Prueba.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Determino que la Regresión Logística con enunciado completo fue el mejor modelo generalista
mejor_modelo = lr_full
test_x_asociada = testEmb_full

# Reporto la exactitud
print('Test-accuracy con el mejor modelo: %.2f%%' % (100 * mejor_modelo.score(test_x_asociada, y_test)))

# Realizo la predicción
predicciones = mejor_modelo.predict(test_x_asociada)

# Construyo e imprimo las métricas
print('\nMatriz de Confusión:')
print(confusion_matrix(y_test, predicciones, labels=[0,1]))

print('\nMatriz de Confusión (Proporciones):')
print(confusion_matrix(y_test, predicciones, labels=[0,1]) / predicciones.shape[0])

print('\nReporte Final de Clasificación:')
print(classification_report(y_test, predicciones))

# *********** Aquí termina la sección de agregar código *************

Análisis del Mejor Modelo:

Basado en las métricas de validación, se seleccionó la Regresión Logística entrenada sobre los vectores del enunciado completo como el modelo definitivo. La evaluación sobre el conjunto de prueba permite revisar su desempeño con datos que no fueron utilizados durante el entrenamiento ni durante la selección del modelo.

La matriz de confusión y el reporte de clasificación muestran cuántos comentarios positivos y negativos fueron clasificados correctamente, así como los falsos positivos y falsos negativos. Estas métricas son importantes porque permiten evaluar no solo la exactitud global, sino también el equilibrio del modelo entre ambas clases.

# **Pregunta - 12:**

Incluye tus comentarios finales de la actividad.

### ++++++++ Inicia la sección de agregar texto: +++++++++++
Comentarios y Conclusiones Finales de la Actividad:

A partir de los resultados obtenidos en el análisis de este corpus (Amazon, Yelp, IMDb), se consolidan los siguientes aprendizajes:

1. El valor del orden secuencial en NLP: representar un texto promediando los vectores de sus palabras es una técnica funcional, pero sacrifica parte de la gramática, el orden y el contexto. Al utilizar un modelo preentrenado de embeddings de HuggingFace sobre el enunciado completo, se puede conservar mejor la carga semántica general del comentario.

2. La sinergia entre modelos densos y algoritmos lineales: no siempre se requieren algoritmos predictivos complejos si los datos de entrada son informativos. Un modelo como Regresión Logística puede ofrecer buenos resultados cuando trabaja con embeddings de calidad.

3. La importancia del preprocesamiento y la partición de datos: separar correctamente los conjuntos de entrenamiento, validación y prueba ayuda a evaluar la capacidad real de generalización de los modelos. Además, mantener el balance entre clases permite que la comparación sea más justa.

4. Comparación entre estrategias de representación: los vectores promedio de palabras permiten una primera aproximación al análisis de sentimiento, pero los vectores del enunciado completo suelen conservar mejor el significado global de la opinión.

En conclusión, esta actividad permitió comprobar cómo los comentarios pueden transformarse en representaciones numéricas útiles para entrenar modelos de clasificación de sentimiento. También mostró que la calidad de la representación del texto influye directamente en el desempeño final del modelo.

### ++++++++ Termina la sección de agregar texto: +++++++++++

# **Fin de la Actividad de Vectores Embebidos - HuggingFace**